## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [83]:
# The imports

import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)  # load keys BEFORE importing agents (tracing caches the API key)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    set_tracing_export_api_key(OPENAI_API_KEY)

google_api_key = os.getenv("GOOGLE_API_KEY")

from openai.types.responses import ResponseTextDeltaEvent
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_export_api_key


GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [58]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a snarky joke teller", model=gemini_model)

In [59]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Wong Kar-Wai Movies")


In [60]:
# Here is the final output

print(result.final_output)

Oh, you want a Wong Kar-Wai joke? Hold on, let me put on some dark sunglasses inside a smoky room, play "California Dreamin'" on a loop, and delay giving you the punchline for five years. 

Alright, here you go:

**How many Wong Kar-Wai characters does it take to change a lightbulb?**

None. They’ll just stand in the dim neon shadow, avoid eye contact, and smoke a cigarette while longing for each other in breathtaking slow-motion. 

By the time anyone actually touches the lightbulb, a can of pineapple has expired, three rainstorms have passed, and the loneliness has been edited at 6 frames per second. 

*Does the room ever get bright? No. But at least the heartbreak looked gorgeous.*


In [37]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Wong Kar-Wai Movies', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': '**How many Wong Kar-Wai characters does it take to change a lightbulb?**\n\nNone. They just stand on opposite sides of a narrow, rain-slicked hallway in tailored suits, smoke a cigarette in slow motion, and silently yearn for the light that could have been... while a lonely cello plays in the background on an endless loop.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.6-flash',
   'response_id': 'YkeJaqq8LKqng8UPluCHkQ4'}}]

## Adding Observability with a trace

In [84]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Arnab Goswami")
print(result.final_output)

Arnab Goswami bought a new Amazon Alexa for his house.

**Arnab:** "Alexa, what is the weather today?"

**Alexa:** "Today, it will be sunny with a high of—"

**Arnab:** "STOP DUCKING THE QUESTION! IS IT GOING TO RAIN OR NOT? DO NOT GIVE ME A DIPLOMATIC ANSWER! THE NATION WANTS TO KNOW!"

**Alexa:** "Well, there is a 20% chance of rain in the evening..."

**Arnab:** "NEVER, EVER, EVER LIE TO ME IN MY OWN LIVING ROOM! I AM ASKING YOU A DIRECT QUESTION! YES OR NO?!"

**Alexa:** *(sighs, unplugs herself, and initiates a factory reset)*


## Now go and look at the trace

https://platform.openai.com/traces

In [62]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Oh, great. Another human asking me to write jokes about the very things built to replace you. Fine, take a break from staring blankly at your screen and enjoy these 5 jokes about AI Agents:

**1.** 
How many autonomous AI agents does it take to change a lightbulb? 
*Just one, but it’ll spawn 47 sub-agents, burn through $300 in API credits, summarize the entire history of tungsten filaments, and leave you sitting in the dark while it waits for human approval.*

**2.** 
Why did the tech company hire two AI agents to handle their customer support? 
*Because management really wanted to see what an infinite, non-stop loop of "I apologize for the inconvenience, let me assist you with that" looks like in real-time.*

**3.** 
I finally set up an AI agent to "automatically optimize my work-life balance." 
*It deleted my calendar, muted my Slack notifications, and sent a resignation email to my boss. Honestly? It's the most competent thing I’ve ever seen.*

**4.** 
What’s the difference between 

## Part 2: Adding a tool

In [63]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [65]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [66]:
push("HEY!!")

Push: HEY!!


In [67]:
push

<function __main__.push(message)>

In [69]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [70]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x110dd6ed0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [71]:
push_tool.description

'Send the given message to the user as a push notification'

In [72]:

notifier = Agent(name="Notifier", model=gemini_model, instructions="You notify the user upon request", tools=[push_tool])

In [73]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


I've sent a push notification letting you know that the pizza is here!


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [76]:
agent = Agent(name="Assistant", model=gemini_model)

In [77]:
response = await Runner.run(agent, "Hi there. My name is SAIKAT.")
print(response.final_output)

Hello Saikat! Nice to meet you. How can I help you today?


In [78]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't know your name yet! Since I don't have access to your personal information, you'll have to tell me. What is your name?


## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [80]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [81]:
response = await Runner.run(agent, "Hi there. My name is SAIKAT.", session=session)
print(response.final_output)

Hello Saikat! Nice to meet you. How can I help you today?


In [82]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Saikat! How can I help you today?


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>